In [1]:
import qibo
from qibo import gates
from qibo.models import Circuit

In [2]:
c = Circuit(6)
c.add(gates.H(0))
c.add(gates.CNOT(0, 1))
c.add(gates.X(0).controlled_by(1,2,3,4,5))

In [3]:
c.draw()

0: ─H─o─X─
1: ───X─o─
2: ─────o─
3: ─────o─
4: ─────o─
5: ─────o─


In [6]:
from qibo import Circuit, gates

c = Circuit(8)
c.add(gates.H(0))
c.add(gates.CNOT(0, 1))

mcx = gates.X(0).controlled_by(1, 2, 3, 4, 5)

# 使用 qubit 6,7 作为辅助比特（这里只是示意）
decomposed = mcx.decompose(6, 7, use_toffolis=True)

for g in decomposed:
    c.add(g)

c.draw()


0:     ─H─o─o─RY─X─RY─X─RY─X─RY─────────────RY─X─RY─X─RY─X─RY─o─RY─X─RY─X─RY─ ...
1:     ───X─|────|────|────|─────────o─────────|────|────|────|────|────|──── ...
2:     ─────|────|────|────|────o────|────o────|────|────|────|────|────|──── ...
3:     ─────|────|────o────|────|────|────|────|────o────|────|────|────o──── ...
4:     ─────o────|─────────|────|────|────|────|─────────|────o────|───────── ...
5:     ─────|────o─────────o─RY─X─RY─X─RY─X─RY─o─────────o────|────o───────── ...
6:     ─────X─────────────────────────────────────────────────X────────────── ...
7:     ────────────────────────────────────────────────────────────────────── ...

0: ... X─RY─────────────RY─X─RY─X─RY─X─RY─H─X─TDG─X─T─X─TDG─X───T───H───o─RY─ ...
1: ... |─────────o─────────|────|────|──────|─────|───|─────|───────────|──── ...
2: ... |────o────|────o────|────|────|──────|─────|───|─────|───────────|──── ...
3: ... |────|────|────|────|────o────|──────|─────|───|─────|───────────|──── ...
4: ... |────|──

In [10]:
qasm_str = c.to_qasm()

In [11]:
with open('my_circuit.qasm', 'w') as f:
    f.write(qasm_str)

print("QASM文件已保存为 'my_circuit.qasm'")

QASM文件已保存为 'my_circuit.qasm'


In [ ]:
# 导入所需的库：matplotlib用于绘图，qibolab用于脉冲控制
import matplotlib.pyplot as plt
# 从qibolab导入脉冲相关的核心类
from qibolab import Pulse, PulseSequence, Readout, Acquisition, Delay
# 从qibolab导入脉冲形状（包络）类
from qibolab import Gaussian, Rectangular, Drag


# ==========================================
# 1. 定义物理参数
# ==========================================
# 定义量子比特的驱动频率，单位为赫兹（4.5 GHz）
qubit_freq = 4.5e9    
# 定义读取谐振腔的频率，单位为赫兹（7.2 GHz）
readout_freq = 7.2e9  

# ==========================================
# 2. 构建驱动脉冲 (Drive Pulse)
# ==========================================
# 修复说明：移除了start参数，因为新版qibolab不支持在Pulse构造函数中直接指定起始时间
# 创建一个高斯形状的驱动脉冲，用于操控量子比特状态
drive_pulse = Pulse(
    duration=40,                                    # 脉冲持续时间，单位为纳秒（ns）
    amplitude=0.6,                                 # 脉冲幅度，归一化到[-1, 1]范围
    relative_phase=0,                               # 脉冲的相对相位，单位为弧度
    envelope=Gaussian(rel_sigma=0.2)                 # 脉冲包络形状，使用高斯函数，rel_sigma=0.2表示标准差为持续时间的20%
)

# ==========================================
# 3. 构建读取脉冲 (Readout Pulse)
# ==========================================
# 在新版qibolab API中，读取操作通常由两部分组成：一个探测脉冲（Probe Pulse）和一个采集窗口（Acquisition）
# 修复说明：移除了frequency参数，因为Pulse类本身不支持直接设置频率，频率通常在通道配置中定义
# 创建一个Readout对象，用于读取量子比特状态
ro_pulse = Readout(
    acquisition=Acquisition(duration=2000),            # 采集窗口：定义信号采集的时间长度，单位为纳秒
    probe=Pulse(                                     # 探测脉冲：发送到读取谐振腔的激励信号
        duration=2000,                               # 探测脉冲持续时间，单位为纳秒
        amplitude=0.8,                               # 探测脉冲幅度
        relative_phase=0,                               # 探测脉冲相位
        envelope=Rectangular()                           # 使用矩形（方波）包络，适合读取操作
    )
)

# ==========================================
# 4. 组装序列 (The Sequence Container)
# ==========================================
# 修复说明：使用PulseSequence构造函数直接创建序列，而不是使用不存在的add()方法
# 创建脉冲序列，将驱动脉冲和读取脉冲按顺序添加到相应的通道
sequence = PulseSequence([
    ("0/drive", drive_pulse),                     # 将驱动脉冲添加到量子比特0的驱动通道
    ("0/readout", ro_pulse)                       # 将读取脉冲添加到量子比特0的读取通道
])

# ==========================================
# 5. 审计与可视化
# ==========================================
# 修复说明：直接使用duration属性计算总时间，因为PulseSequence对象没有finish属性
# 计算整个脉冲序列的总持续时间（驱动脉冲持续时间 + 读取脉冲持续时间）
total_duration = drive_pulse.duration + ro_pulse.probe.duration
# 打印序列总持续时间
print(f"Total Sequence Duration: {total_duration} ns")

# 打印序列审计标题
print("\n--- Sequence Audit ---")
# 简化的序列审计，避免复杂的类型检查问题，直接访问已知的属性
# 打印驱动脉冲的详细信息
print(f"Drive Pulse: Duration = {drive_pulse.duration}ns, Shape = {type(drive_pulse.envelope).__name__}")
# 打印读取脉冲的详细信息（通过ro_pulse.probe访问探测脉冲属性）
print(f"Readout Pulse: Duration = {ro_pulse.probe.duration}ns, Shape = {type(ro_pulse.probe.envelope).__name__}")
# 打印序列中包含的元素总数
print(f"Sequence contains {len(sequence)} elements")

# 可选功能：如果安装了绘图库，可以尝试绘制脉冲序列的可视化图形
# sequence.plot()  # 取消注释以启用绘图功能